# 17 Transformer自监督预训练

**目的**：严格使用16号冻结的外折索引，训练并冻结两类谱图编码器：

- **B路线**：32,000张公共背景实测谱；
- **E路线**：公共背景实测谱＋该外折允许的非稀疏CFM谱。

每个外折独立训练、独立选择epoch、独立保存检查点。本Notebook不读取PBT标签、不做PBT预测、
不读取外折真实评价谱，也不读取ChemVista。

## 训练目标与边界

1. **Masked peak modeling**：随机遮蔽15%的有效峰，重建四维峰token。
2. **Same-structure contrastive learning**：同一二维InChIKey的两张谱或同一谱的两次独立遮蔽作为正对。
3. B路线的正对来自公共库同结构实测谱；E路线另加入同结构的不同CFM能量/模式谱。
4. `is_synthetic`继续作为条件输入；当前冻结背景池与CFM身份完全不重叠，因此本阶段不伪造跨域同身份正对。
5. 每条路线内部按二维InChIKey哈希固定90%训练、10%自监督验证；验证集不参与梯度。
6. 最佳epoch只按自监督验证损失选择，不读取任何PBT端点或外折测试性能。
7. 正式结果为5折×2路线=10个最佳检查点；中断后可从每个run的last checkpoint继续。

In [ ]:
from __future__ import annotations

from contextlib import nullcontext
from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import os
import platform
import random
import sys
import time
import warnings

# Must be set before the first CUDA BLAS call to make the requested
# deterministic CUDA behavior effective on Windows as well.
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
# NumPy only handles light indexing in this notebook; keeping its MKL
# layer sequential avoids loading a second Intel OpenMP runtime beside
# the CUDA-enabled PyTorch runtime in some Windows conda environments.
os.environ.setdefault('MKL_THREADING_LAYER', 'SEQUENTIAL')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 260)
plt.style.use('seaborn-v0_8-whitegrid')

# -------------------- 用户运行配置 --------------------
RUN_TRAINING = True
RETRAIN_COMPLETED = False
RESUME_INCOMPLETE = True
REQUIRE_CUDA = False
DRY_RUN = False
# ----------------------------------------------------

# -------------------- 冻结训练参数 --------------------
ROUTES_TO_PRETRAIN = ['B_real_pretrain', 'E_cfm_pretrain_real_finetune']
N_OUTER_FOLDS = 5
BASE_SEED = 20260906
MAX_EPOCHS = 6
MIN_EPOCHS = 3
EARLY_STOPPING_PATIENCE = 2
BATCH_SIZE = 64
NUM_WORKERS = 0  # Windows/Jupyter最稳定
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
MASK_PROBABILITY = 0.15
CONTRASTIVE_TEMPERATURE = 0.10
RECONSTRUCTION_WEIGHT = 1.0
CONTRASTIVE_WEIGHT = 0.10
MAX_GRAD_NORM = 1.0
USE_MIXED_PRECISION = True
VALIDATION_HASH_MODULUS = 10
VALIDATION_HASH_REMAINDER = 0

MODEL_CONFIG = {
    'peak_input_dim': 4,
    'metadata_numeric_dim': 8,
    'd_model': 128,
    'n_heads': 8,
    'n_layers': 4,
    'dim_feedforward': 256,
    'dropout': 0.10,
    'projection_dim': 128,
}
# ----------------------------------------------------

if DRY_RUN:
    MAX_EPOCHS = 1
    MIN_EPOCHS = 1
    EARLY_STOPPING_PATIENCE = 1
    BATCH_SIZE = 16

cwd = Path.cwd().resolve()
roots = [cwd, *cwd.parents, Path(r'E:\谱图预测PBT_终版')]
ROOT = next((path for path in roots if (path / 'outputs' / 'pbt_msms_v1').exists()), None)
assert ROOT is not None, '无法定位项目根目录，请手动设置ROOT。'

BASE = ROOT / 'outputs' / 'pbt_msms_v1'
FREEZE_DIR = BASE / 'transformer_data_v1'
OUTPUT_DIR = BASE / 'transformer_pretraining_v1'
if DRY_RUN:
    OUTPUT_DIR = BASE / 'transformer_pretraining_v1_dryrun'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
FIGURE_DIR = OUTPUT_DIR / 'figures'
for path in (OUTPUT_DIR, CHECKPOINT_DIR, FIGURE_DIR):
    path.mkdir(parents=True, exist_ok=True)

PATHS = {
    'freeze_manifest': FREEZE_DIR / 'transformer_data_freeze_manifest.json',
    'freeze_qc': FREEZE_DIR / 'transformer_data_freeze_qc.csv',
    'token_schema': FREEZE_DIR / 'transformer_token_schema.json',
    'vocab': FREEZE_DIR / 'transformer_categorical_vocab.json',
    'background_index': FREEZE_DIR / 'real_background_pretraining_index.csv.gz',
    'cfm_index': FREEZE_DIR / 'cfm_transformer_spectrum_index.csv.gz',
    'background_cache': FREEZE_DIR / 'token_caches' / 'real_background_tokens.npz',
    'cfm_cache': FREEZE_DIR / 'token_caches' / 'cfm_tokens.npz',
}
missing = [str(path) for path in PATHS.values() if not path.exists()]
assert not missing, '缺少17号输入文件：\n' + '\n'.join(missing)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if REQUIRE_CUDA:
    assert device.type == 'cuda', 'REQUIRE_CUDA=True，但当前PyTorch无法使用CUDA。'
amp_enabled = bool(USE_MIXED_PRECISION and device.type == 'cuda')

print('ROOT:', ROOT)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('PyTorch:', torch.__version__)
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
print('AMP:', amp_enabled, 'DRY_RUN:', DRY_RUN)

## 1. 验证16号冻结结果

检查16号状态、全部QC和每个待使用文件的SHA256。任何缓存或外折索引变化都会停止，
防止在训练过程中悄悄改变数据。

In [ ]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def as_bool(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({'true', '1', 'yes', 'y', 't'})

def atomic_csv(frame, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temporary, index=False, encoding='utf-8-sig')
    os.replace(temporary, path)

def atomic_json(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    os.replace(temporary, path)

def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode('utf-8')
    return int(hashlib.sha256(payload).hexdigest()[:8], 16)

def stable_partition(key, modulus=VALIDATION_HASH_MODULUS):
    value = int(hashlib.sha256(str(key).encode('utf-8')).hexdigest()[:16], 16)
    return value % modulus

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

freeze = read_json(PATHS['freeze_manifest'])
freeze_qc = pd.read_csv(PATHS['freeze_qc'])
token_schema = read_json(PATHS['token_schema'])
vocab = read_json(PATHS['vocab'])
assert freeze['status'] == 'READY_FOR_NOTEBOOK_17'
assert freeze['external_test_unlocked'] is False
assert freeze['chemvista_spectra_used'] is False
assert as_bool(freeze_qc['passed']).all()
assert token_schema['max_peaks'] == 120
assert len(token_schema['peak_token_columns']) == MODEL_CONFIG['peak_input_dim']
assert len(token_schema['metadata_numeric_columns']) == MODEL_CONFIG['metadata_numeric_dim']

for relative_path, expected_hash in freeze['output_hashes'].items():
    candidate = FREEZE_DIR / Path(relative_path)
    if candidate.exists() and candidate.is_file():
        actual_hash = sha256_file(candidate)
        assert actual_hash == expected_hash, f'16号冻结文件发生变化：{relative_path}'

background_index = pd.read_csv(PATHS['background_index'])
cfm_index = pd.read_csv(PATHS['cfm_index'])
assert len(background_index) == freeze['counts']['real_background_spectra']
assert len(cfm_index) == freeze['counts']['cfm_cached_spectra']
assert set(background_index['domain']) == {'real'}
assert set(cfm_index['domain']) == {'synthetic'}

fold_input_paths = {}
for outer_fold in range(freeze['fold_policy']['n_outer_folds']):
    for route in ROUTES_TO_PRETRAIN:
        path = FREEZE_DIR / 'fold_indices' / f'outer_{outer_fold}' / f'{route}__pretraining.csv.gz'
        assert path.exists()
        expected_hash = freeze['output_hashes'][str(path.relative_to(FREEZE_DIR))]
        assert sha256_file(path) == expected_hash
        fold_input_paths[(outer_fold, route)] = path

print('Freeze status:', freeze['status'])
print('Background spectra:', len(background_index))
print('CFM spectra cached:', len(cfm_index))
print('Pretraining index files:', len(fold_input_paths))

## 2. 载入冻结token缓存

缓存全部载入内存，但每个batch才转换成GPU张量。两个缓存总计约9 MB，适合RTX 4060 Ti。

In [ ]:
class TokenStore:
    def __init__(self, cache_paths):
        self.arrays = {}
        for cache_name, path in cache_paths.items():
            with np.load(path) as loaded:
                self.arrays[cache_name] = {name: loaded[name] for name in loaded.files}
            required = {'peak_tokens', 'padding_mask', 'metadata_numeric', 'adduct_code', 'instrument_code', 'source_code'}
            assert required <= set(self.arrays[cache_name])

    def get(self, cache_name, row):
        arrays = self.arrays[str(cache_name)]
        index = int(row)
        return {
            'peaks': arrays['peak_tokens'][index].astype(np.float32),
            'padding': arrays['padding_mask'][index].astype(bool),
            'metadata': arrays['metadata_numeric'][index].astype(np.float32),
            'adduct': int(arrays['adduct_code'][index]),
            'instrument': int(arrays['instrument_code'][index]),
            'source': int(arrays['source_code'][index]),
        }

store = TokenStore({
    'real_background_tokens.npz': PATHS['background_cache'],
    'cfm_tokens.npz': PATHS['cfm_cache'],
})
print({name: arrays['peak_tokens'].shape for name, arrays in store.arrays.items()})

## 3. 同结构谱对数据集

每个epoch每个二维InChIKey只贡献一个谱对，避免多谱图化合物支配训练。若同一身份只有一张谱，
对同一谱实施两次独立masked-peak视图；若有多张谱，则无放回选择两张。

In [ ]:
class StructurePairDataset(Dataset):
    def __init__(self, index_frame, token_store, training, seed, mask_probability=MASK_PROBABILITY):
        self.frame = index_frame.reset_index(drop=True).copy()
        self.store = token_store
        self.training = bool(training)
        self.seed = int(seed)
        self.mask_probability = float(mask_probability)
        self.groups = {
            key: group.index.to_numpy(dtype=int)
            for key, group in self.frame.groupby('inchikey_2d', sort=True)
        }
        self.keys = sorted(self.groups)
        self.epoch = 0

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

    def __len__(self):
        return len(self.keys)

    def _masked_view(self, frame_index, rng):
        row = self.frame.iloc[int(frame_index)]
        sample = self.store.get(row.cache_name, row.cache_row)
        peaks = sample['peaks'].copy()
        padding = sample['padding'].copy()
        valid_indices = np.flatnonzero(~padding)
        assert len(valid_indices) >= 3, f'主预训练索引含少于3峰谱：{row.sample_id}'
        mask_n = max(1, int(round(len(valid_indices) * self.mask_probability)))
        mask_n = min(mask_n, len(valid_indices) - 1)
        masked_indices = rng.choice(valid_indices, size=mask_n, replace=False)
        target = peaks.copy()
        mask_flags = np.zeros(len(peaks), dtype=np.float32)
        mask_flags[masked_indices] = 1.0
        peaks[masked_indices] = 0.0
        return {
            'peaks': torch.from_numpy(peaks),
            'target': torch.from_numpy(target),
            'padding': torch.from_numpy(padding),
            'mask_flags': torch.from_numpy(mask_flags),
            'metadata': torch.from_numpy(sample['metadata']),
            'adduct': torch.tensor(sample['adduct'], dtype=torch.long),
            'instrument': torch.tensor(sample['instrument'], dtype=torch.long),
            'source': torch.tensor(sample['source'], dtype=torch.long),
            'domain': torch.tensor(1 if str(row.source_pool) == 'cfm' else 0, dtype=torch.long),
        }

    def __getitem__(self, index):
        key = self.keys[int(index)]
        epoch_component = self.epoch if self.training else 0
        rng = np.random.default_rng(stable_seed(self.seed, epoch_component, index, key))
        candidates = self.groups[key]
        if len(candidates) >= 2:
            chosen = rng.choice(candidates, size=2, replace=False)
        else:
            chosen = np.repeat(candidates[0], 2)
        return self._masked_view(chosen[0], rng), self._masked_view(chosen[1], rng)

def split_pretraining_index(frame):
    frame = frame.copy()
    frame['self_supervised_split'] = np.where(
        frame['inchikey_2d'].map(stable_partition).eq(VALIDATION_HASH_REMAINDER),
        'validation', 'train',
    )
    train = frame.loc[frame['self_supervised_split'].eq('train')].copy()
    validation = frame.loc[frame['self_supervised_split'].eq('validation')].copy()
    assert set(train['inchikey_2d']).isdisjoint(set(validation['inchikey_2d']))
    assert len(train) and len(validation)
    if DRY_RUN:
        train_keys = sorted(train['inchikey_2d'].unique())[:32]
        validation_keys = sorted(validation['inchikey_2d'].unique())[:16]
        train = train.loc[train['inchikey_2d'].isin(train_keys)].copy()
        validation = validation.loc[validation['inchikey_2d'].isin(validation_keys)].copy()
    return train, validation

## 4. 条件化Set Transformer编码器

峰序列不使用位置编码，因此在padding mask固定时对峰排列保持集合不变性。谱图条件通过CLS token注入；
加合物、仪器类别、来源类别均使用独立embedding，整数编码不存在大小顺序含义。

In [ ]:
class SpectrumSetTransformer(nn.Module):
    def __init__(self, config, vocab_sizes):
        super().__init__()
        d_model = config['d_model']
        self.peak_projection = nn.Sequential(
            nn.Linear(config['peak_input_dim'] + 1, d_model),
            nn.GELU(), nn.LayerNorm(d_model),
        )
        self.metadata_projection = nn.Sequential(
            nn.Linear(config['metadata_numeric_dim'], d_model),
            nn.GELU(), nn.LayerNorm(d_model),
        )
        self.adduct_embedding = nn.Embedding(vocab_sizes['adduct'], d_model)
        self.instrument_embedding = nn.Embedding(vocab_sizes['instrument_category'], d_model)
        self.source_embedding = nn.Embedding(vocab_sizes['source_library'], d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=config['n_heads'],
            dim_feedforward=config['dim_feedforward'], dropout=config['dropout'],
            activation='gelu', batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=config['n_layers'])
        self.final_norm = nn.LayerNorm(d_model)
        self.reconstruction_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, config['peak_input_dim']),
        )
        self.projection_head = nn.Sequential(
            nn.Linear(d_model, d_model), nn.GELU(),
            nn.Linear(d_model, config['projection_dim']),
        )
        nn.init.normal_(self.cls_token, std=0.02)

    def forward(self, peaks, padding, mask_flags, metadata, adduct, instrument, source):
        peak_input = torch.cat([peaks, mask_flags.unsqueeze(-1)], dim=-1)
        peak_tokens = self.peak_projection(peak_input)
        condition = (
            self.metadata_projection(metadata)
            + self.adduct_embedding(adduct)
            + self.instrument_embedding(instrument)
            + self.source_embedding(source)
        )
        cls = self.cls_token.expand(len(peaks), -1, -1) + condition.unsqueeze(1)
        tokens = torch.cat([cls, peak_tokens], dim=1)
        cls_padding = torch.zeros((len(peaks), 1), dtype=torch.bool, device=padding.device)
        full_padding = torch.cat([cls_padding, padding], dim=1)
        encoded = self.final_norm(self.encoder(tokens, src_key_padding_mask=full_padding))
        spectrum_embedding = F.normalize(self.projection_head(encoded[:, 0]), dim=-1)
        reconstructed_peaks = self.reconstruction_head(encoded[:, 1:])
        return spectrum_embedding, reconstructed_peaks

def parameter_count(model):
    return int(sum(parameter.numel() for parameter in model.parameters()))

VOCAB_SIZES = {
    'adduct': len(vocab['adduct']),
    'instrument_category': len(vocab['instrument_category']),
    'source_library': len(vocab['source_library']),
}
example_model = SpectrumSetTransformer(MODEL_CONFIG, VOCAB_SIZES)
print('Trainable parameters:', parameter_count(example_model))
del example_model

## 5. 损失函数、训练和验证

重建损失只在被遮蔽峰计算；对比损失为batch内双向InfoNCE。验证阶段的谱对选择和遮蔽完全固定，
因而不同epoch可直接比较。

In [ ]:
def move_view(view, target_device):
    return {key: value.to(target_device, non_blocking=True) for key, value in view.items()}

def reconstruction_loss(prediction, target, mask_flags):
    selected = mask_flags.bool()
    assert selected.any()
    difference = F.smooth_l1_loss(prediction[selected], target[selected], reduction='none')
    feature_weights = torch.tensor([2.0, 1.0, 1.0, 1.0], device=difference.device)
    return (difference * feature_weights).mean()

def contrastive_loss(embedding_a, embedding_b, temperature=CONTRASTIVE_TEMPERATURE):
    logits = embedding_a @ embedding_b.T / temperature
    labels = torch.arange(len(embedding_a), device=embedding_a.device)
    loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))
    accuracy = 0.5 * (
        (logits.argmax(dim=1) == labels).float().mean()
        + (logits.argmax(dim=0) == labels).float().mean()
    )
    return loss, accuracy

def forward_pair(model, view_a, view_b):
    embedding_a, reconstruction_a = model(
        view_a['peaks'], view_a['padding'], view_a['mask_flags'], view_a['metadata'],
        view_a['adduct'], view_a['instrument'], view_a['source'],
    )
    embedding_b, reconstruction_b = model(
        view_b['peaks'], view_b['padding'], view_b['mask_flags'], view_b['metadata'],
        view_b['adduct'], view_b['instrument'], view_b['source'],
    )
    reconstruction = 0.5 * (
        reconstruction_loss(reconstruction_a, view_a['target'], view_a['mask_flags'])
        + reconstruction_loss(reconstruction_b, view_b['target'], view_b['mask_flags'])
    )
    contrastive, retrieval_accuracy = contrastive_loss(embedding_a, embedding_b)
    total = RECONSTRUCTION_WEIGHT * reconstruction + CONTRASTIVE_WEIGHT * contrastive
    return total, reconstruction, contrastive, retrieval_accuracy, embedding_a, embedding_b

def make_loader(dataset, shuffle, seed):
    generator = torch.Generator()
    generator.manual_seed(int(seed))
    return DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS,
        pin_memory=device.type == 'cuda', drop_last=shuffle and len(dataset) >= BATCH_SIZE,
        generator=generator,
    )

def run_epoch(model, loader, optimizer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    totals = {'total_loss': 0.0, 'reconstruction_loss': 0.0, 'contrastive_loss': 0.0, 'retrieval_accuracy': 0.0}
    n_examples = 0
    embedding_chunks = []
    iterator = tqdm(loader, leave=False, desc='train' if training else 'validation')
    for view_a, view_b in iterator:
        view_a = move_view(view_a, device)
        view_b = move_view(view_b, device)
        batch_n = len(view_a['peaks'])
        if training:
            optimizer.zero_grad(set_to_none=True)
        context = torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled)
        with torch.set_grad_enabled(training), context:
            total, reconstruction, contrastive, retrieval, emb_a, emb_b = forward_pair(model, view_a, view_b)
        if training:
            scaler.scale(total).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
        totals['total_loss'] += float(total.detach()) * batch_n
        totals['reconstruction_loss'] += float(reconstruction.detach()) * batch_n
        totals['contrastive_loss'] += float(contrastive.detach()) * batch_n
        totals['retrieval_accuracy'] += float(retrieval.detach()) * batch_n
        n_examples += batch_n
        if not training and sum(len(chunk) for chunk in embedding_chunks) < 4096:
            embedding_chunks.append(torch.cat([emb_a, emb_b], dim=0).detach().float().cpu())
        iterator.set_postfix(loss=f'{float(total.detach()):.4f}')
    assert n_examples > 0
    result = {key: value / n_examples for key, value in totals.items()}
    if embedding_chunks:
        embeddings = torch.cat(embedding_chunks, dim=0)
        result['embedding_mean_feature_std'] = float(embeddings.std(dim=0).mean())
        # Mean over every pairwise cosine equals the squared norm of the
        # mean normalized embedding.  This avoids materializing an NxN
        # BLAS product and prevents a Windows OpenMP runtime collision.
        result['embedding_mean_pairwise_cosine'] = float(embeddings.mean(dim=0).square().sum())
    else:
        result['embedding_mean_feature_std'] = np.nan
        result['embedding_mean_pairwise_cosine'] = np.nan
    return result

## 6. 按外折训练B/E检查点

每个run完成一个epoch即保存`last`；验证损失改善时更新`best`。已完成run默认跳过，
`RETRAIN_COMPLETED=True`才会明确重训。

In [ ]:
HISTORY_PATH = OUTPUT_DIR / 'transformer_pretraining_history.csv'
RUN_RECORD_PATH = OUTPUT_DIR / 'transformer_pretraining_run_records.csv'
history = pd.read_csv(HISTORY_PATH).to_dict('records') if HISTORY_PATH.exists() else []
run_records = pd.read_csv(RUN_RECORD_PATH).to_dict('records') if RUN_RECORD_PATH.exists() else []

def update_unique_record(records, new_record, keys=('outer_fold', 'route')):
    filtered = [row for row in records if not all(str(row.get(key)) == str(new_record.get(key)) for key in keys)]
    filtered.append(new_record)
    return filtered

def checkpoint_metadata(outer_fold, route, input_path):
    return {
        'outer_fold': int(outer_fold), 'route': route,
        'freeze_manifest_sha256': sha256_file(PATHS['freeze_manifest']),
        'pretraining_index_sha256': sha256_file(input_path),
        'model_config': MODEL_CONFIG, 'vocab_sizes': VOCAB_SIZES,
        'training_config': {
            'max_epochs': MAX_EPOCHS, 'min_epochs': MIN_EPOCHS,
            'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
            'weight_decay': WEIGHT_DECAY, 'mask_probability': MASK_PROBABILITY,
            'contrastive_temperature': CONTRASTIVE_TEMPERATURE,
            'reconstruction_weight': RECONSTRUCTION_WEIGHT,
            'contrastive_weight': CONTRASTIVE_WEIGHT,
        },
    }

def save_torch_atomic(payload, path):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    os.replace(temporary, path)

folds_to_run = [0] if DRY_RUN else list(range(N_OUTER_FOLDS))
for outer_fold in folds_to_run:
    fold_checkpoint_dir = CHECKPOINT_DIR / f'outer_{outer_fold}'
    fold_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    for route in ROUTES_TO_PRETRAIN:
        input_path = fold_input_paths[(outer_fold, route)]
        best_path = fold_checkpoint_dir / f'{route}__best.pt'
        last_path = fold_checkpoint_dir / f'{route}__last.pt'
        existing_record = next((
            row for row in run_records
            if int(row.get('outer_fold', -1)) == outer_fold and str(row.get('route')) == route
            and str(row.get('status')) == 'completed'
        ), None)
        expected_metadata = checkpoint_metadata(outer_fold, route, input_path)
        if existing_record and best_path.exists() and not RETRAIN_COMPLETED:
            saved = torch.load(best_path, map_location='cpu', weights_only=False)
            assert saved['metadata'] == expected_metadata, f'已完成检查点配置不一致：{best_path}'
            print('Skip completed:', outer_fold, route)
            continue
        if not RUN_TRAINING:
            print('RUN_TRAINING=False, skip:', outer_fold, route)
            continue

        pretraining_index = pd.read_csv(input_path)
        assert set(pretraining_index['cache_name']) <= set(store.arrays)
        assert set(pretraining_index['source_pool']) <= ({'background_unlabeled_real'} if route == 'B_real_pretrain' else {'background_unlabeled_real', 'cfm'})
        train_index, validation_index = split_pretraining_index(pretraining_index)
        assert set(train_index['inchikey_2d']).isdisjoint(set(validation_index['inchikey_2d']))
        run_seed = stable_seed(BASE_SEED, outer_fold, route)
        set_global_seed(run_seed)
        train_dataset = StructurePairDataset(train_index, store, True, run_seed)
        validation_dataset = StructurePairDataset(validation_index, store, False, run_seed + 1)
        validation_loader = make_loader(validation_dataset, False, run_seed + 2)

        model = SpectrumSetTransformer(MODEL_CONFIG, VOCAB_SIZES).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
        scaler = torch.amp.GradScaler('cuda', enabled=amp_enabled)
        start_epoch = 0
        best_epoch = -1
        best_validation_loss = np.inf
        stale_epochs = 0
        elapsed_before = 0.0

        if last_path.exists() and RESUME_INCOMPLETE and not RETRAIN_COMPLETED:
            saved = torch.load(last_path, map_location=device, weights_only=False)
            if saved.get('metadata') == expected_metadata:
                model.load_state_dict(saved['model_state'])
                optimizer.load_state_dict(saved['optimizer_state'])
                scheduler.load_state_dict(saved['scheduler_state'])
                scaler.load_state_dict(saved['scaler_state'])
                start_epoch = int(saved['epoch']) + 1
                best_epoch = int(saved['best_epoch'])
                best_validation_loss = float(saved['best_validation_loss'])
                stale_epochs = int(saved['stale_epochs'])
                elapsed_before = float(saved.get('elapsed_seconds', 0.0))
                print('Resume:', outer_fold, route, 'from epoch', start_epoch)

        started = time.perf_counter()
        completed_epoch = start_epoch - 1
        for epoch in range(start_epoch, MAX_EPOCHS):
            train_dataset.set_epoch(epoch)
            train_loader = make_loader(train_dataset, True, run_seed + epoch * 1009)
            train_metrics = run_epoch(model, train_loader, optimizer=optimizer, scaler=scaler)
            validation_metrics = run_epoch(model, validation_loader)
            scheduler.step()
            completed_epoch = epoch
            row = {
                'outer_fold': outer_fold, 'route': route, 'epoch': epoch,
                'train_group_n': len(train_dataset), 'validation_group_n': len(validation_dataset),
                'learning_rate': optimizer.param_groups[0]['lr'],
                **{f'train_{key}': value for key, value in train_metrics.items()},
                **{f'validation_{key}': value for key, value in validation_metrics.items()},
            }
            history = [item for item in history if not (
                int(item.get('outer_fold', -1)) == outer_fold
                and str(item.get('route')) == route
                and int(item.get('epoch', -1)) == epoch
            )]
            history.append(row)
            atomic_csv(pd.DataFrame(history).sort_values(['outer_fold', 'route', 'epoch']), HISTORY_PATH)

            improved = validation_metrics['total_loss'] < best_validation_loss - 1e-8
            if improved:
                best_validation_loss = float(validation_metrics['total_loss'])
                best_epoch = epoch
                stale_epochs = 0
                save_torch_atomic({
                    'metadata': expected_metadata, 'epoch': epoch,
                    'model_state': {key: value.detach().cpu() for key, value in model.state_dict().items()},
                    'validation_metrics': validation_metrics,
                }, best_path)
            else:
                stale_epochs += 1

            elapsed = elapsed_before + time.perf_counter() - started
            save_torch_atomic({
                'metadata': expected_metadata, 'epoch': epoch,
                'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict(),
                'scheduler_state': scheduler.state_dict(), 'scaler_state': scaler.state_dict(),
                'best_epoch': best_epoch, 'best_validation_loss': best_validation_loss,
                'stale_epochs': stale_epochs, 'elapsed_seconds': elapsed,
            }, last_path)
            print(
                f'fold={outer_fold} route={route} epoch={epoch} '
                f'train={train_metrics["total_loss"]:.5f} val={validation_metrics["total_loss"]:.5f} '
                f'best={best_validation_loss:.5f}'
            )
            if epoch + 1 >= MIN_EPOCHS and stale_epochs >= EARLY_STOPPING_PATIENCE:
                break

        assert best_path.exists()
        best_saved = torch.load(best_path, map_location='cpu', weights_only=False)
        best_metrics = best_saved['validation_metrics']
        total_elapsed = elapsed_before + time.perf_counter() - started
        record = {
            'outer_fold': outer_fold, 'route': route, 'status': 'completed',
            'run_seed': run_seed, 'train_spectra_n': len(train_index),
            'train_compound_n': train_index['inchikey_2d'].nunique(),
            'validation_spectra_n': len(validation_index),
            'validation_compound_n': validation_index['inchikey_2d'].nunique(),
            'best_epoch': int(best_saved['epoch']),
            'epochs_completed': completed_epoch + 1,
            'best_validation_total_loss': float(best_metrics['total_loss']),
            'best_validation_reconstruction_loss': float(best_metrics['reconstruction_loss']),
            'best_validation_contrastive_loss': float(best_metrics['contrastive_loss']),
            'best_validation_retrieval_accuracy': float(best_metrics['retrieval_accuracy']),
            'best_validation_embedding_mean_feature_std': float(best_metrics['embedding_mean_feature_std']),
            'runtime_minutes': total_elapsed / 60.0,
            'best_checkpoint': str(best_path.relative_to(OUTPUT_DIR)),
            'best_checkpoint_sha256': sha256_file(best_path),
            'input_index_sha256': sha256_file(input_path),
            'device': str(device),
        }
        run_records = update_unique_record(run_records, record)
        atomic_csv(pd.DataFrame(run_records).sort_values(['outer_fold', 'route']), RUN_RECORD_PATH)

run_records_frame = pd.read_csv(RUN_RECORD_PATH) if RUN_RECORD_PATH.exists() else pd.DataFrame()
history_frame = pd.read_csv(HISTORY_PATH) if HISTORY_PATH.exists() else pd.DataFrame()
display(run_records_frame)

## 7. 训练诊断与检查点QC

检查所有run的有限损失、embedding非完全坍塌、输入身份隔离、检查点元数据和哈希。
自监督loss只用于训练稳定性诊断，不代表PBT预测性能。

In [ ]:
expected_runs = len(folds_to_run) * len(ROUTES_TO_PRETRAIN)
completed_records = run_records_frame.loc[
    run_records_frame['outer_fold'].isin(folds_to_run)
    & run_records_frame['route'].isin(ROUTES_TO_PRETRAIN)
    & run_records_frame['status'].eq('completed')
].copy() if len(run_records_frame) else pd.DataFrame()

integrity_rows = []
for outer_fold in folds_to_run:
    b_index = pd.read_csv(fold_input_paths[(outer_fold, 'B_real_pretrain')])
    e_index = pd.read_csv(fold_input_paths[(outer_fold, 'E_cfm_pretrain_real_finetune')])
    b_train, b_val = split_pretraining_index(b_index)
    e_train, e_val = split_pretraining_index(e_index)
    integrity_rows.append({
        'outer_fold': outer_fold,
        'b_sources_correct': set(b_index['source_pool']) == {'background_unlabeled_real'},
        'e_sources_correct': set(e_index['source_pool']) == {'background_unlabeled_real', 'cfm'},
        'b_train_validation_identity_overlap': len(set(b_train['inchikey_2d']) & set(b_val['inchikey_2d'])),
        'e_train_validation_identity_overlap': len(set(e_train['inchikey_2d']) & set(e_val['inchikey_2d'])),
        'b_cfm_n': int(b_index['source_pool'].eq('cfm').sum()),
        'e_cfm_n': int(e_index['source_pool'].eq('cfm').sum()),
        'b_spectra_n': len(b_index), 'e_spectra_n': len(e_index),
    })
integrity = pd.DataFrame(integrity_rows)
atomic_csv(integrity, OUTPUT_DIR / 'transformer_pretraining_input_integrity.csv')

checkpoint_metadata_ok = True
checkpoint_hash_ok = True
if len(completed_records):
    for row in completed_records.itertuples(index=False):
        checkpoint_path = OUTPUT_DIR / row.best_checkpoint
        if not checkpoint_path.exists() or sha256_file(checkpoint_path) != row.best_checkpoint_sha256:
            checkpoint_hash_ok = False
            continue
        saved = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
        expected = checkpoint_metadata(int(row.outer_fold), str(row.route), fold_input_paths[(int(row.outer_fold), str(row.route))])
        checkpoint_metadata_ok &= saved.get('metadata') == expected

finite_columns = [
    'best_validation_total_loss', 'best_validation_reconstruction_loss',
    'best_validation_contrastive_loss', 'best_validation_retrieval_accuracy',
    'best_validation_embedding_mean_feature_std',
]
finite_metrics = bool(
    len(completed_records) == expected_runs
    and np.isfinite(completed_records[finite_columns].to_numpy(float)).all()
)
noncollapsed = bool(
    len(completed_records) == expected_runs
    and (completed_records['best_validation_embedding_mean_feature_std'].astype(float) > 1e-4).all()
)

qc = pd.DataFrame([
    {'check': 'freeze_ready', 'passed': freeze['status'] == 'READY_FOR_NOTEBOOK_17', 'detail': freeze['status']},
    {'check': 'chemvista_remains_locked', 'passed': freeze['external_test_unlocked'] is False and freeze['chemvista_spectra_used'] is False, 'detail': 'no ChemVista paths loaded'},
    {'check': 'all_expected_runs_completed', 'passed': len(completed_records) == expected_runs, 'detail': f'{len(completed_records)}/{expected_runs}'},
    {'check': 'all_best_metrics_finite', 'passed': finite_metrics, 'detail': str(finite_columns)},
    {'check': 'embedding_not_completely_collapsed', 'passed': noncollapsed, 'detail': '>1e-4 mean feature std'},
    {'check': 'checkpoint_hashes_match_records', 'passed': checkpoint_hash_ok and len(completed_records) == expected_runs, 'detail': len(completed_records)},
    {'check': 'checkpoint_metadata_matches_frozen_inputs', 'passed': checkpoint_metadata_ok and len(completed_records) == expected_runs, 'detail': len(completed_records)},
    {'check': 'B_route_contains_real_background_only', 'passed': integrity['b_sources_correct'].all() and integrity['b_cfm_n'].eq(0).all(), 'detail': len(integrity)},
    {'check': 'E_route_contains_real_background_and_cfm', 'passed': integrity['e_sources_correct'].all() and integrity['e_cfm_n'].gt(0).all(), 'detail': integrity['e_cfm_n'].tolist()},
    {'check': 'self_supervised_train_validation_identity_disjoint', 'passed': integrity[['b_train_validation_identity_overlap', 'e_train_validation_identity_overlap']].eq(0).all().all(), 'detail': 'IK2D hash split'},
    {'check': 'no_pbt_label_file_loaded', 'passed': not any('label' in key.lower() for key in PATHS), 'detail': 'pretraining is label-free'},
])
atomic_csv(qc, OUTPUT_DIR / 'transformer_pretraining_qc.csv')
display(integrity)
display(qc)

if len(history_frame):
    figure_data = history_frame.loc[
        history_frame['outer_fold'].isin(folds_to_run)
        & history_frame['route'].isin(ROUTES_TO_PRETRAIN)
    ].copy()
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
    route_colors = {
        'B_real_pretrain': '#2878B5',
        'E_cfm_pretrain_real_finetune': '#F28522',
    }
    metrics = [
        ('train_total_loss', 'Self-supervised training loss'),
        ('validation_total_loss', 'Frozen self-supervised validation loss'),
        ('validation_retrieval_accuracy', 'Same-structure retrieval accuracy'),
    ]
    first_fold = min(folds_to_run)
    for axis, (metric, title) in zip(axes, metrics):
        for (outer_fold, route), group in figure_data.groupby(['outer_fold', 'route'], sort=True):
            group = group.sort_values('epoch')
            axis.plot(
                group['epoch'], group[metric], marker='o', linewidth=1.5, alpha=0.78,
                color=route_colors[route],
                label=route if int(outer_fold) == first_fold else '_nolegend_',
            )
        axis.set_xlabel('Epoch')
        axis.set_title(title)
        axis.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / '17_pretraining_diagnostics.png', dpi=240, bbox_inches='tight')
    plt.close(fig)
    print('Diagnostic figure saved:', FIGURE_DIR / '17_pretraining_diagnostics.png')

## 8. 最终运行清单

正式运行必须完成10个检查点且全部QC通过。17号只冻结编码器，不选择最终PBT模型；
B/E对PBT是否有帮助必须由18号的真实谱外折测试决定。

In [ ]:
all_qc_pass = bool(as_bool(qc['passed']).all())
if DRY_RUN:
    PRETRAINING_STATUS = 'DRY_RUN_VALIDATION_ONLY'
elif not RUN_TRAINING and len(completed_records) < expected_runs:
    PRETRAINING_STATUS = 'PRETRAINING_INCOMPLETE_RESUME'
elif all_qc_pass:
    PRETRAINING_STATUS = 'READY_FOR_NOTEBOOK_18'
else:
    PRETRAINING_STATUS = 'BLOCKED_REVIEW_QC'

formal_outputs = [
    HISTORY_PATH, RUN_RECORD_PATH,
    OUTPUT_DIR / 'transformer_pretraining_input_integrity.csv',
    OUTPUT_DIR / 'transformer_pretraining_qc.csv',
    FIGURE_DIR / '17_pretraining_diagnostics.png',
    *sorted(CHECKPOINT_DIR.glob('outer_*/*__best.pt')),
]
formal_outputs = [path for path in formal_outputs if path.exists()]
manifest = {
    'notebook': '17_Transformer自监督预训练.ipynb',
    'generated_at_utc': utc_now(),
    'status': PRETRAINING_STATUS,
    'external_test_unlocked': False,
    'chemvista_used': False,
    'pbt_labels_used': False,
    'scientific_scope': 'Fold-specific B/E self-supervised spectral encoder pretraining only',
    'device': {
        'type': str(device),
        'gpu': torch.cuda.get_device_name(0) if device.type == 'cuda' else None,
        'mixed_precision': amp_enabled,
    },
    'model': {**MODEL_CONFIG, 'parameter_count': parameter_count(SpectrumSetTransformer(MODEL_CONFIG, VOCAB_SIZES))},
    'objectives': {
        'masked_peak_probability': MASK_PROBABILITY,
        'masked_peak_reconstruction_weight': RECONSTRUCTION_WEIGHT,
        'same_structure_infonce_weight': CONTRASTIVE_WEIGHT,
        'temperature': CONTRASTIVE_TEMPERATURE,
        'domain_condition': 'is_synthetic retained in frozen metadata input',
        'cross_domain_same_identity_pairs': False,
    },
    'self_supervised_validation': {
        'method': 'stable IK2D SHA256 split',
        'modulus': VALIDATION_HASH_MODULUS,
        'validation_remainder': VALIDATION_HASH_REMAINDER,
        'best_epoch_metric': 'validation_total_loss',
    },
    'training': {
        'routes': ROUTES_TO_PRETRAIN,
        'outer_folds': folds_to_run,
        'max_epochs': MAX_EPOCHS, 'minimum_epochs': MIN_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'batch_size_in_structure_pairs': BATCH_SIZE,
        'optimizer': 'AdamW', 'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY, 'max_grad_norm': MAX_GRAD_NORM,
        'completed_runs': len(completed_records), 'expected_runs': expected_runs,
    },
    'input_hashes': {
        'freeze_manifest': sha256_file(PATHS['freeze_manifest']),
        'background_cache': sha256_file(PATHS['background_cache']),
        'cfm_cache': sha256_file(PATHS['cfm_cache']),
        **{
            f'outer_{fold}_{route}': sha256_file(path)
            for (fold, route), path in fold_input_paths.items()
            if fold in folds_to_run
        },
    },
    'output_hashes': {
        str(path.relative_to(OUTPUT_DIR)): sha256_file(path) for path in formal_outputs
    },
    'run_records': completed_records.to_dict('records'),
    'software': {
        'python': sys.version, 'torch': torch.__version__,
        'numpy': np.__version__, 'pandas': pd.__version__,
        'platform': platform.platform(),
    },
}
atomic_json(manifest, OUTPUT_DIR / 'transformer_pretraining_manifest.json')
print('PRETRAINING_STATUS:', PRETRAINING_STATUS)
print('Manifest:', OUTPUT_DIR / 'transformer_pretraining_manifest.json')
if PRETRAINING_STATUS == 'READY_FOR_NOTEBOOK_18':
    print('17号完成，可以进入18号P/B/T微调与A–F消融。')
elif PRETRAINING_STATUS == 'DRY_RUN_VALIDATION_ONLY':
    print('这只是DRY_RUN；改回DRY_RUN=False后完整运行。')
else:
    print('尚未形成全部正式检查点；保持配置不变重新运行可续跑。')

## 下一步

18号加载每个外折对应的B/E最佳编码器，完成A–F六条路线、至少3个随机种子的P/B/T masked多任务微调。
所有性能只在16号冻结的真实Development外层测试谱上计算，再按化合物聚合并做配对bootstrap。
ChemVista仍保持锁定，直到19号完成内部路线选择和阈值冻结。